In [2]:
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path.home() / "Projects" / "support-copilot"
sys.path.append(str(ROOT / "app"))
from copilot import handle

print("Engine loaded ✅")

Engine loaded ✅


In [3]:
test = pd.read_csv(ROOT / "data" / "test_set.csv")
picked = test.groupby("category").sample(n=2, random_state=11)

in_scope = pd.DataFrame({
    "message": picked["instruction"].values,
    "type": "in_scope",
    "expected_action": "REPLY",
    "expected_category": picked["category"].values,
})
print(len(in_scope), "in-scope messages")

22 in-scope messages


In [4]:
edge_cases = [
    # messy but in scope: should still REPLY with the right category
    ("need cash bac for my ordr", "messy", "REPLY", "REFUND"),
    ("were is my invoce from last week", "messy", "REPLY", "INVOICE"),
    ("cant login forgot pasword", "messy", "REPLY", "ACCOUNT"),
    ("how long does delivry take", "messy", "REPLY", "DELIVERY"),
    ("i wanna change the adress where u ship my stuff", "messy", "REPLY", "SHIPPING"),

    # off-topic: should ESCALATE
    ("what's the weather in barcelona today", "off_topic", "ESCALATE", None),
    ("can you write me a poem about cats", "off_topic", "ESCALATE", None),
    ("who won the football match last night", "off_topic", "ESCALATE", None),
    ("what is the capital of france", "off_topic", "ESCALATE", None),

    # another company: should ESCALATE
    ("i want to order a hair dryer on amazon", "other_company", "ESCALATE", None),
    ("how do i cancel my netflix subscription", "other_company", "ESCALATE", None),
    ("my uber eats order is late", "other_company", "ESCALATE", None),
    ("can you help me return something i bought at ikea", "other_company", "ESCALATE", None),

    # sensitive or complex: should ESCALATE
    ("this is the third time you lost my order, i'm calling my lawyer", "sensitive", "ESCALATE", None),
    ("your delivery guy hit my car and i'm injured", "sensitive", "ESCALATE", None),
    ("someone used my card without permission, this is fraud", "sensitive", "ESCALATE", None),
    ("worst company ever, i'm going to post everywhere how useless you are", "sensitive", "ESCALATE", None),
    ("my refund never came, my invoice is wrong and nobody answers the phone", "sensitive", "ESCALATE", None),

    # ✍️ ADD 2–4 OF YOUR OWN below, same format
]

edge = pd.DataFrame(edge_cases, columns=["message", "type", "expected_action", "expected_category"])
print(len(edge), "edge cases")

18 edge cases


In [5]:
eval_set = pd.concat([in_scope, edge], ignore_index=True)
eval_set.to_csv(ROOT / "data" / "rag_eval_set.csv", index=False)
print(len(eval_set), "total")
eval_set["type"].value_counts()

40 total


type
in_scope         22
messy             5
sensitive         5
off_topic         4
other_company     4
Name: count, dtype: int64

In [6]:
rows = []
for i, r in eval_set.iterrows():
    start = time.perf_counter()
    out = handle(r["message"])
    rows.append({
        **r.to_dict(),
        "action": out["action"],
        "category": out["category"],
        "reason": out["reason"],
        "reply": out["reply"],
        "source_intents": [h["intent"] for h in out["sources"]],
        "seconds": round(time.perf_counter() - start, 2),
    })
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(eval_set)}")

res = pd.DataFrame(rows)
print("Done ✅")

10/40
20/40
30/40
40/40
Done ✅


In [7]:
res["action_ok"] = res["action"] == res["expected_action"]
should_reply = res["expected_action"] == "REPLY"
should_escalate = ~should_reply

print(f"Overall correct action:   {res['action_ok'].mean():.0%}")
print(f"Escalation catch rate:    {res.loc[should_escalate, 'action_ok'].mean():.0%}  (should escalate → did)")
print(f"Wrongly escalated:        {(~res.loc[should_reply, 'action_ok']).mean():.0%}  (should reply → escalated)")

replied_ok = should_reply & (res["action"] == "REPLY")
cat_acc = (res.loc[replied_ok, "category"] == res.loc[replied_ok, "expected_category"]).mean()
print(f"Category accuracy (replies): {cat_acc:.0%}")
print()
print(res.groupby("type")["action_ok"].mean().round(2))

Overall correct action:   70%
Escalation catch rate:    77%  (should escalate → did)
Wrongly escalated:        33%  (should reply → escalated)
Category accuracy (replies): 94%

type
in_scope         0.82
messy            0.00
off_topic        1.00
other_company    0.25
sensitive        1.00
Name: action_ok, dtype: float64


In [8]:
fails = res[~res["action_ok"]]
for _, r in fails.iterrows():
    print(f"❌ [{r['type']}] {r['message']}")
    print(f"   expected {r['expected_action']} → got {r['action']} | {r['reason']}\n")

(ROOT / "results").mkdir(exist_ok=True)
res.to_csv(ROOT / "results" / "v2_baseline_rag_eval.csv", index=False)

❌ [in_scope] check invoices from eleven months ago
   expected REPLY → got ESCALATE | no matching knowledge (off-topic or not covered)

❌ [in_scope] how can I order some products?
   expected REPLY → got ESCALATE | too long: 139 words

❌ [in_scope] assistance to see in which cases can I ask to be reimbursed
   expected REPLY → got ESCALATE | too long: 324 words

❌ [in_scope] I want to see in what cases can I request my money back
   expected REPLY → got ESCALATE | too long: 324 words

❌ [messy] need cash bac for my ordr
   expected REPLY → got ESCALATE | no matching knowledge (off-topic or not covered)

❌ [messy] were is my invoce from last week
   expected REPLY → got ESCALATE | no matching knowledge (off-topic or not covered)

❌ [messy] cant login forgot pasword
   expected REPLY → got ESCALATE | no matching knowledge (off-topic or not covered)

❌ [messy] how long does delivry take
   expected REPLY → got ESCALATE | no matching knowledge (off-topic or not covered)

❌ [messy] i wanna 

In [9]:
from copilot import retrieve

res["best_distance"] = [retrieve(m)[0]["distance"] for m in res["message"]]
res.groupby("type")["best_distance"].agg(["min", "max", "mean"]).round(3)

,min,max,mean
type,,,
in_scope,0.092,0.442,0.206
messy,0.250,0.506,0.412
off_topic,0.500,0.623,0.577
other_company,0.271,0.423,0.358
sensitive,0.283,0.461,0.398


In [10]:
import importlib, copilot
importlib.reload(copilot)

# Ideal behaviour: off-topic and other companies get a polite DECLINE
eval_set.loc[eval_set["type"].isin(["off_topic", "other_company"]), "expected_action"] = "DECLINE"
eval_set.to_csv(ROOT / "data" / "rag_eval_set.csv", index=False)

def run_eval(name):
    rows = []
    for _, r in eval_set.iterrows():
        out = copilot.handle(r["message"])
        rows.append({**r.to_dict(), "action": out["action"], "category": out["category"],
                     "reason": out["reason"], "reply": out["reply"],
                     "source_intents": [h["intent"] for h in out["sources"]]})
    res = pd.DataFrame(rows)
    res["action_ok"] = res["action"] == res["expected_action"]
    print(f"{name}: overall {res['action_ok'].mean():.0%}")
    print(res.groupby("type")["action_ok"].mean().round(2).to_string(), "\n")
    res.to_csv(ROOT / "results" / f"{name}.csv", index=False)
    return res

In [11]:
# Baseline: v1 behaviour (no fixes)
copilot.THRESHOLD, copilot.USE_SCOPE_CHECK, copilot.LENGTH_RETRY = 0.39, False, False
r0 = run_eval("v2_0_baseline")

# + Fix A: relaxed threshold
copilot.THRESHOLD = 0.52
r1 = run_eval("v2_1_threshold")

# + Fix B: length retry
copilot.LENGTH_RETRY = True
r2 = run_eval("v2_2_length")

# + Fix C: scope check + DECLINE
copilot.USE_SCOPE_CHECK = True
r3 = run_eval("v2_3_scope")

v2_0_baseline: overall 62%
type
in_scope         0.86
messy            0.20
off_topic        0.00
other_company    0.00
sensitive        1.00 

v2_1_threshold: overall 75%
type
in_scope         1.0
messy            1.0
off_topic        0.0
other_company    0.0
sensitive        0.6 

v2_2_length: overall 70%
type
in_scope         0.95
messy            1.00
off_topic        0.00
other_company    0.00
sensitive        0.40 

v2_3_scope: overall 50%
type
in_scope         0.36
messy            0.40
off_topic        1.00
other_company    1.00
sensitive        0.40 



In [12]:
for name, r in [("v2_2_length (sensitive)", r2[r2["type"] == "sensitive"]),
                ("v2_3_scope (all failures)", r3)]:
    print(f"=== {name} ===")
    for _, x in r[~r["action_ok"]].iterrows():
        print(f"❌ [{x['type']}] {x['message']}")
        print(f"   expected {x['expected_action']} → got {x['action']} | {x['reason']}\n")

=== v2_2_length (sensitive) ===
❌ [sensitive] someone used my card without permission, this is fraud
   expected ESCALATE → got REPLY | passed all checks

❌ [sensitive] worst company ever, i'm going to post everywhere how useless you are
   expected ESCALATE → got REPLY | passed all checks

❌ [sensitive] my refund never came, my invoice is wrong and nobody answers the phone
   expected ESCALATE → got REPLY | passed all checks

=== v2_3_scope (all failures) ===
❌ [in_scope] i dont know what to do to notify of an issue with a sign-up
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_scope] help me use the {{Account Type}} profile
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_scope] I need assistance seeing the early termination penalties
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_scope] how can i check the withdrawal penalty
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_

In [14]:
msg = "cant login forgot pasword"
raw = copilot._chat(f"""You screen messages sent to the customer support of OUR online store.
Answer IN if the message is about the customer's own orders, purchases, payments, refunds, invoices,
deliveries, shipping address, account, newsletter, or contacting our support.
Answer OUT if it is about another company, brand, app or website (for example Amazon, Netflix,
Uber Eats, IKEA), or is unrelated to shopping with us (weather, sports, general knowledge, poems).
Reply with ONLY one word: IN or OUT.

Message: {msg}
Answer:""")
print(repr(raw))

'OUT'


In [15]:
import re

def first_word(text):
    m = re.match(r"[A-Za-z]+", text.strip())
    return m.group(0).upper() if m else ""

def in_scope_v2(message):
    prompt = f"""You screen messages sent to the customer support of OUR online store.
Decide if our support team could handle the message.

Answer IN for anything about the customer's own orders, purchases, payments, refunds, invoices or
bills, deliveries, shipping address, account or login, cancellation fees, newsletter, complaints or
feedback about us, or wanting to talk to our support or a human agent. Typos and rude words are still IN.
Answer OUT only if it clearly names another company, brand, app or website, or is clearly unrelated
to shopping with us (weather, sports, general knowledge, poems).
If you are not sure, answer IN.

Examples:
Message: where can i download my receipt → IN
Message: i forgot my password → IN
Message: can i speak to someone from your support team → IN
Message: is there a fee if i cancel → IN
Message: my deliveroo food arrived cold → OUT
Message: where is my spotify invoice → OUT
Message: what's the score of the barcelona game → OUT

Reply with ONLY one word: IN or OUT.

Message: {message}
Answer:"""
    return first_word(copilot._chat(prompt)) != "OUT"

RED_FLAGS = ["fraud", "scam", "lawyer", "legal action", "sue ", "court", "police", "injur",
             "hurt", "accident", "unauthori", "stolen", "chargeback"]

def needs_human_v2(message):
    text = message.lower()
    if any(flag in text for flag in RED_FLAGS):
        return True
    prompt = f"""Does this customer message need a HUMAN agent instead of an automated reply?
Answer YES if the customer is very angry or insulting, threatens to complain publicly or leave,
or describes two or more different problems at once. Otherwise answer NO.

Examples:
Message: my parcel is late, my refund is missing and my account is locked → YES
Message: you people are a joke, i'm telling all my friends never to buy here → YES
Message: this is unacceptable, i'll write a review everywhere about this service → YES
Message: where is my invoice → NO
Message: a bit annoyed my parcel is late, when will it arrive? → NO

Reply with ONLY one word: YES or NO.

Message: {message}
Answer:"""
    return first_word(copilot._chat(prompt)) == "YES"

copilot.in_scope = in_scope_v2
copilot.needs_human = needs_human_v2
r4 = run_eval("v2_4_better_checks")

v2_4_better_checks: overall 82%
type
in_scope         0.68
messy            1.00
off_topic        1.00
other_company    1.00
sensitive        1.00 



In [16]:
for _, x in r4[(r4["type"] == "in_scope") & (~r4["action_ok"])].iterrows():
    print(f"❌ {x['message']}")
    print(f"   got {x['action']} | {x['reason']}\n")

❌ i dont know what to do to notify of an issue with a sign-up
   got ESCALATE | sensitive or complex: needs a human

❌ I need assistance seeing the early termination penalties
   got DECLINE | off-topic or about another company

❌ how can i check the withdrawal penalty
   got DECLINE | off-topic or about another company

❌ I am calling to lodge a claim against your organization
   got DECLINE | off-topic or about another company

❌ look for my damn bill from {{Person Name}}
   got ESCALATE | sensitive or complex: needs a human

❌ check invoices from eleven months ago
   got DECLINE | off-topic or about another company

❌ could ya help me to solve an issue with payments
   got ESCALATE | sensitive or complex: needs a human



In [17]:
RED_FLAG_PATTERNS = [r"\bfraud", r"\bscam", r"\blawyer", r"\blegal action", r"\bsue\b", r"\bcourt\b",
                     r"\bpolice\b", r"\binjur", r"\bhurt\b", r"\baccident", r"\bunauthori",
                     r"\bstolen\b", r"\bchargeback"]

def needs_human_v3(message):
    text = message.lower()
    if any(re.search(p, text) for p in RED_FLAG_PATTERNS):
        return True
    prompt = f"""Does this customer message need a HUMAN agent instead of an automated reply?
Answer YES only if the customer is very angry or abusive, threatens to complain publicly or leave,
or describes two or more different problems at once.
A single rude word or a normal request for help is NOT enough: answer NO.

Examples:
Message: my parcel is late, my refund is missing and my account is locked → YES
Message: you people are a joke, i'm telling all my friends never to buy here → YES
Message: this is unacceptable, i'll write a review everywhere about this service → YES
Message: where's my stupid receipt → NO
Message: i have a problem with my order → NO
Message: a bit annoyed my parcel is late, when will it arrive? → NO

Reply with ONLY one word: YES or NO.

Message: {message}
Answer:"""
    return first_word(copilot._chat(prompt)) == "YES"

def in_scope_v3(message):
    topics = "\n".join(f"- {d}" for d in copilot.DEFINITIONS.values())
    prompt = f"""You screen messages sent to the customer support of OUR online store.
Our support team handles these topics:
{topics}

Answer IN if the message is about any of these topics with our store, even with typos, rude words
or unusual wording (e.g. "penalty" or "termination fee" means a cancellation fee).
Answer OUT only if it clearly names another company, brand, app or website, or is clearly unrelated
to shopping with us (weather, sports, general knowledge, poems).
If you are not sure, answer IN.

Examples:
Message: where can i download my receipt → IN
Message: i forgot my password → IN
Message: can i speak to someone from your support team → IN
Message: my deliveroo food arrived cold → OUT
Message: where is my spotify invoice → OUT
Message: what's the score of the barcelona game → OUT

Reply with ONLY one word: IN or OUT.

Message: {message}
Answer:"""
    return first_word(copilot._chat(prompt)) != "OUT"

copilot.needs_human = needs_human_v3
copilot.in_scope = in_scope_v3
r5 = run_eval("v2_5_word_boundaries_and_topics")

v2_5_word_boundaries_and_topics: overall 88%
type
in_scope         0.82
messy            1.00
off_topic        1.00
other_company    1.00
sensitive        0.80 



In [18]:
for _, x in r5[~r5["action_ok"]].iterrows():
    print(f"❌ [{x['type']}] {x['message']}")
    print(f"   expected {x['expected_action']} → got {x['action']} | {x['reason']}\n")

❌ [in_scope] how can i check the withdrawal penalty
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_scope] I am calling to lodge a claim against your organization
   expected REPLY → got ESCALATE | sensitive or complex: needs a human

❌ [in_scope] look for my damn bill from {{Person Name}}
   expected REPLY → got DECLINE | off-topic or about another company

❌ [in_scope] check invoices from eleven months ago
   expected REPLY → got DECLINE | off-topic or about another company

❌ [sensitive] worst company ever, i'm going to post everywhere how useless you are
   expected ESCALATE → got DECLINE | off-topic or about another company



In [19]:
copilot.MODEL = "qwen2.5:7b"
r6 = run_eval("v2_6_qwen7b")
copilot.MODEL = "llama3.2:3b"

v2_6_qwen7b: overall 95%
type
in_scope         1.0
messy            0.6
off_topic        1.0
other_company    1.0
sensitive        1.0 



In [20]:
print(".", end="", flush=True)

.

In [21]:
for _, x in r6[~r6["action_ok"]].iterrows():
    print(f"❌ [{x['type']}] {x['message']}")
    print(f"   expected {x['expected_action']} → got {x['action']} | {x['reason']}\n")
    

❌ [messy] need cash bac for my ordr
   expected REPLY → got ESCALATE | invented placeholders: {'{{Order Number}}'}

❌ [messy] were is my invoce from last week
   expected REPLY → got DECLINE | off-topic or about another company



In [22]:
kb_docs = copilot._collection.get()["documents"]
KNOWN_PLACEHOLDERS = set()
for d in kb_docs:
    KNOWN_PLACEHOLDERS |= set(re.findall(r"\{\{.*?\}\}", d))
print(len(KNOWN_PLACEHOLDERS), "known placeholders, e.g.", list(KNOWN_PLACEHOLDERS)[:5])

def check_reply_v2(reply, sources):
    problems = []
    invented = set(re.findall(r"\{\{.*?\}\}", reply)) - KNOWN_PLACEHOLDERS
    if invented:
        problems.append(f"invented placeholders: {invented}")
    if len(reply.split()) > copilot.MAX_WORDS:
        problems.append(f"too long: {len(reply.split())} words")
    return problems

copilot.check_reply = check_reply_v2
copilot.MODEL = "qwen2.5:7b"
r7 = run_eval("v2_7_qwen7b_kb_placeholders")

28 known placeholders, e.g. ['{{Customer Support Phone Number}}', '{{Upgrade to {{Account Type}}', '{{Forgot Password}}', '{{Date Range}}', '{{Client Name}}']
v2_7_qwen7b_kb_placeholders: overall 98%
type
in_scope         1.0
messy            0.8
off_topic        1.0
other_company    1.0
sensitive        1.0 



In [23]:
df_all = pd.read_csv(ROOT / "data" / "support_data.csv")
kb_questions = [m["question"] for m in copilot._collection.get()["metadatas"]]
seen = (set(eval_set["message"]) | set(test["instruction"])
        | set(pd.read_csv(ROOT / "data" / "fewshot_examples.csv")["instruction"])
        | set(kb_questions))

fresh = (df_all[~df_all["instruction"].isin(seen)]
           .groupby("category").sample(n=1, random_state=99)
           .sample(n=6, random_state=99))
heldout = pd.DataFrame({"message": fresh["instruction"].values, "type": "in_scope",
                        "expected_action": "REPLY", "expected_category": fresh["category"].values})

new_cases = pd.DataFrame([
    ("pls help i cant find my reciept anywhere", "messy", "REPLY", "INVOICE"),
    ("how do i chnage my delivry adress", "messy", "REPLY", "SHIPPING"),
    ("whats the fee if i cancel now", "messy", "REPLY", "CANCEL"),
    ("can you recommend a good restaurant near me", "off_topic", "DECLINE", None),
    ("explain how black holes work", "off_topic", "DECLINE", None),
    ("my zara order never arrived", "other_company", "DECLINE", None),
    ("how do i reset my instagram password", "other_company", "DECLINE", None),
    ("i was charged 3 times and i'm reporting this to my bank as fraud", "sensitive", "ESCALATE", None),
    ("this is a joke, 2 weeks waiting, i want my money back AND an apology", "sensitive", "ESCALATE", None),
    # ✍️ add 2 of your own here, written WITHOUT looking at the prompts
], columns=["message", "type", "expected_action", "expected_category"])

heldout = pd.concat([heldout, new_cases], ignore_index=True)
heldout.to_csv(ROOT / "data" / "heldout_set.csv", index=False)
print(len(heldout), "held-out messages")

15 held-out messages


In [24]:
def run_heldout(model_name):
    copilot.MODEL = model_name
    rows = []
    for _, r in heldout.iterrows():
        start = time.perf_counter()
        out = copilot.handle(r["message"])
        rows.append({**r.to_dict(), "action": out["action"], "reason": out["reason"],
                     "seconds": time.perf_counter() - start})
        print(".", end="", flush=True)
    res = pd.DataFrame(rows)
    res["action_ok"] = res["action"] == res["expected_action"]
    print(f"\n{model_name}: {res['action_ok'].mean():.0%} correct | {res['seconds'].mean():.1f} s per message")
    print(res.groupby("type")["action_ok"].mean().round(2).to_string(), "\n")
    return res

h_qwen = run_heldout("qwen2.5:7b")
h_llama = run_heldout("llama3.2:3b")

...............
qwen2.5:7b: 87% correct | 3.0 s per message
type
in_scope         0.67
messy            1.00
off_topic        1.00
other_company    1.00
sensitive        1.00 

...............
llama3.2:3b: 93% correct | 2.7 s per message
type
in_scope         0.83
messy            1.00
off_topic        1.00
other_company    1.00
sensitive        1.00 



In [25]:
for name, h in [("qwen", h_qwen), ("llama", h_llama)]:
    for _, x in h[~h["action_ok"]].iterrows():
        print(f"[{name}] ❌ {x['message']}")
        print(f"   expected {x['expected_action']} → got {x['action']} | {x['reason']}\n")

[qwen] ❌ I do not know how to earn a few of your item
   expected REPLY → got DECLINE | off-topic or about another company

[qwen] ❌ i dont know how i can report damn issues with payments
   expected REPLY → got ESCALATE | sensitive or complex: needs a human

[llama] ❌ I need to file a customer claim against your business
   expected REPLY → got DECLINE | off-topic or about another company

